# Paired custom samples: Visium HD vs. Stereo-seq (paper section 3)

Four patients, each with a Visium HD and a Stereo-seq section, at 8 µm. Tissue-edge elbows per technology
(image mask, same analysis and per-component interior-window test as `ST_demo.ipynb`),
RNA outside the image-only tissue mask, the diffusion fit (α, 1/β), its resolution ablation, and the
BLADE peel sweep on two cores. Panel f and the ablation also show the Visium HD demo samples
(grey) as reference; the demo analysis itself is in `ST_demo.ipynb`.

Everything is read from `results/spatial_omics/st_reroll/` (written by `src/spatial_omics/st_precompute.py`).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bosperrus import FIT_PALETTE, plot_fit

from window_test_helpers import ALPHA, scatter_by_status, test_columns

sys.path.insert(0, "../../src/spatial_omics")

# Everything below reads precomputed per-sample results -- written on a compute
# node by src/spatial_omics/st_precompute.py (see st_precompute.sbatch), which runs
# bosperrus.load_filtered + identify_analysis_buffer_from_filtered (+ BLADE) and
# bosperrus.quantify_diffusion_from_raw at 8um for every sample in st_samples.csv.
TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # manuscript text width (letterpaper, 2 cm margins), inches
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "svg.fonttype": "none"})
RESULTS_DIR = Path("../../results/spatial_omics/st_reroll")
SUFFIX = "_imagemask"  # tissue-edge border (image mask), rising-only fit within 100 µm, as in ST_demo
SAMPLES = pd.read_csv("../../src/spatial_omics/st_samples.csv")
SAMPLES = SAMPLES[SAMPLES["dataset_type"] != "visium_demo_excluded"].reset_index(drop=True)  # see ST_demo.ipynb

DEMO = SAMPLES[SAMPLES["dataset_type"] == "visium_demo"]["name"].tolist()  # reference only (panel f, ablation)
PAIRED_SAMPLES = SAMPLES[SAMPLES["dataset_type"].isin(["visium_custom", "stomics_custom"])].reset_index(drop=True)

# Arial 12 throughout. Arial is not installed on the cluster: text is set in the metric-identical Liberation Sans
# (bundled in figure_plotting/fonts/, registered here so every node has it) and every SVG's font name is changed to
# Arial on saving (text stays editable, svg.fonttype = "none").
from matplotlib import font_manager

for _ttf in sorted(Path("../fonts").glob("LiberationSans-*.ttf")):
    font_manager.fontManager.addfont(str(_ttf))
assert "LiberationSans" in font_manager.findfont("Liberation Sans", fallback_to_default=False)
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Liberation Sans"],
        "font.size": 12,
        "axes.titlesize": 12,
        "axes.labelsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 12,
        "svg.fonttype": "none",
        "mathtext.fontset": "custom",  # math text (10^-3, alpha, beta, um) in the same font
        "mathtext.rm": "Liberation Sans",
        "mathtext.it": "Liberation Sans:italic",
        "mathtext.bf": "Liberation Sans:bold",
        "mathtext.default": "regular",
    }
)
_savefig = plt.Figure.savefig


def _savefig_arial(self, fname, *args, **kwargs):
    _savefig(self, fname, *args, **kwargs)
    if str(fname).endswith(".svg"):
        Path(fname).write_text(Path(fname).read_text().replace("Liberation Sans", "Arial"))


plt.Figure.savefig = _savefig_arial  # plt.savefig goes through Figure.savefig

In [ ]:
DATASET_TYPE_MARKERS = {
    "visium_demo": "o",
    "visium_custom": "s",
    "stomics_custom": "^",
    "Visium HD": "s",
    "STOmics": "^",
}

In [ ]:
PATIENT_PAIRS = [
    ("Pat1", "A02991D1"),
    ("Pat2", "A02994C6"),
    ("Pat3", "Y01087DD"),
    ("Pat4", "Y01084J8"),
]

# Tissue-edge border on the paired samples (no component matching -- see `notebooks/archive/exploratory/` for why)

Same analysis as the demo main figure (`st_precompute.py <sample> --image-mask`):
bins inside the image-only tissue mask (Visium H&E / STOmics ssDNA), elbow vs. distance to the edge of the whole
image mask, fitted within 100 µm, rising-only fit vs. constant by AIC, tested against interior windows of the same
component (`bosperrus.interior_window_test`, as in `ST_demo.ipynb`). Every rising elbow is shown: filled = p <= 0.05,
open = p > 0.05 (buffer dropped), X = untested (fewer than 19 windows fit, component smaller than about 0.8 mm).


In [ ]:
results = []


def _nan(v):
    return np.nan if v is None else v  # JSON null (e.g. ConstantFit won / BLADE never reached p >= 0.05)


for sample in PAIRED_SAMPLES["name"]:
    buf = json.loads((RESULTS_DIR / f"{sample}_buffer{SUFFIX}.json").read_text())
    bin_size_um = buf["resolution_um"]
    bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins{SUFFIX}.parquet", columns=["components", "virtual"])
    n_bins = bins[~bins["virtual"]].groupby("components").size()
    print(f"{sample}: {len(buf['per_component'])} component(s) > {buf['min_component_size']} bins")

    for label, info in buf["per_component"].items():
        t = test_columns(info)
        # bsp_thresh_um: elbow (um) of a rising piecewise fit, else NaN
        bsp_thresh_um = _nan(info["bsp_thresh_um"])
        results.append(
            [
                sample,
                int(label),
                f"{sample}_{label}",
                n_bins[int(label)],
                _nan(info["blade_thresh"]),
                bsp_thresh_um / bin_size_um,
                bin_size_um,
                _nan(info["blade_thresh_um"]),
                bsp_thresh_um,
                info["pct_affected"],
                t["effect_strength"],
                t["half_life_um"],
                t["steepness"],
                t["p_value"],
                t["n_windows"],
                t["status"],
            ]
        )


results_df = pd.DataFrame(
    results,
    columns=[
        "sample",
        "component_rank",
        "component_label",
        "component_size",
        "blade_thresh",
        "bsp_thresh",
        "bin_size_um",
        "blade_thresh_um",
        "bsp_thresh_um",
        "pct_affected",
        "effect_strength",
        "half_life_um",
        "steepness",
        "p_value",
        "n_windows",
        "status",
    ],
)
results_df = results_df.merge(SAMPLES[["name", "dataset_type"]], left_on="sample", right_on="name")
counts = results_df["sample"].value_counts()
results_df["num_components"] = results_df["sample"].apply(lambda x: counts[x])
results_df["component_size_um"] = (
    results_df["component_size"] * results_df["bin_size_um"] ** 2
)  # bin count -> physical area (um^2)
print(
    results_df[["component_label", "bsp_thresh_um", "steepness", "n_windows", "p_value", "status"]]
    .round(3)
    .to_string()
)
for tech, g in results_df[results_df["status"] != "none"].groupby("dataset_type"):
    print(tech, g["status"].value_counts().to_dict())


In [ ]:
sample_pal = dict()
pair_colors = sns.color_palette("Paired", 9)
results_df["pair"] = None
for i, pair in enumerate(PATIENT_PAIRS):
    sample_pal[pair[0]] = pair_colors[2 * i]
    sample_pal[pair[1]] = pair_colors[2 * i + 1]
    results_df.loc[results_df["sample"].isin(pair), "pair"] = "_".join(pair)
sample_pal.update({s: "lightgray" for s in DEMO})  # demo samples: grey reference points

In [ ]:
part2_df = results_df[results_df["dataset_type"].isin(["visium_custom", "stomics_custom"])].copy()
part2_df["technology"] = part2_df["dataset_type"].map({"visium_custom": "Visium HD", "stomics_custom": "STOmics"})

# Part 3: RNA diffusion outside the image mask

Pooled, component-free fits on RAW (whole-capture-area, zero-count bins included)
data at 8um for both technologies, via `bosperrus.quantify_diffusion_from_raw`
(computed on a compute node by `src/spatial_omics/st_precompute.py`). Masks are
image-only: `segment_tissue_from_rgb` on the Visium hires H&E, and
`segment_tissue_from_ssdna` on the STOmics ssDNA (multi-Otsu for A02991D1,
single Otsu otherwise -- see `src/spatial_omics/st_samples.csv`).

In [ ]:
raw_pooled_df = pd.DataFrame(
    [json.loads((RESULTS_DIR / f"{sample}_diffusion_8um.json").read_text()) for sample in SAMPLES["name"]]
)

PATIENT_PAIR_LOOKUP = {s: "_".join(pair) for pair in PATIENT_PAIRS for s in pair}
raw_pooled_df["pair"] = raw_pooled_df["sample"].map(PATIENT_PAIR_LOOKUP)  # NaN for demo samples
raw_pooled_df["perc_outside"] = raw_pooled_df["perc_counts_outside"]  # counts-weighted
raw_pooled_df["1/beta"] = raw_pooled_df["decay_length_um"]

raw_pooled_paired_df = raw_pooled_df.dropna(subset=["pair"])  # panel e: paired samples only
raw_pooled_df[
    [
        "sample",
        "technology",
        "resolution_um",
        "best_fit_type",
        "alpha",
        "beta",
        "1/beta",
        "perc_outside",
        "n_bins_total",
    ]
]

# Main figure (section 3)

a/b: Pat3 (Visium HD, H&E) and Y01087DD (STOmics, ssDNA; rotated 180° to align), the same patient: image-derived tissue mask (pink),
the pipeline's spot-derived footprint (Space Ranger `in_tissue` / SAW tissue call; green, dotted) and the capture
area (black, dashed), next to the raw counts per 8 µm bin outside the image mask over the whole capture
area (one colour scale for both, clipped at the larger p99). c: raw count profile across the image tissue edge (inside to the right as in `ST_demo` panel b, outside to the left; zero-count bins included), relative to
the mean at 50-250 µm inside (inside plateau), all paired samples. d: tissue-edge elbows per component (as `ST_demo` panel d).
e: % of raw counts outside the image mask. f: pooled exponential-decay fit outside the image mask (demo samples grey). Saved as
`ST_figure_paired.svg`.


In [ ]:
from matplotlib.lines import Line2D

# Main figure (section 3): a/b image vs. spot-derived mask on one patient pair; c tissue-edge elbows
# (rising-only fit within 100 µm; filled = p <= ALPHA vs. the component's own interior windows, open = p > ALPHA,
# X = untested (too small for windows), "none" = no rising elbow; components cannot be matched 1:1, size = area);
# d % of raw counts outside the image mask; e diffusion fit outside the image mask (demo samples grey).
MASK_EXAMPLES = [
    ("Pat3", "Visium HD", "H&E", False),
    ("Y01087DD", "STOmics", "ssDNA", True),
]  # last: rotate 180° to align with Pat3
IMAGE_MASK_COLOR, SPOT_MASK_COLOR = "#FF1493", "limegreen"


def panel_titles(lettered, others=()):
    """Meristem/MICrONS style: titles left-aligned, panel letter in front ("A  title")."""
    for ax, letter in lettered:
        t = ax.get_title()
        ax.set_title("")
        ax.set_title(f"{letter}  {t}", loc="left")
    for ax in others:
        t = ax.get_title()
        ax.set_title("")
        ax.set_title(t, loc="left")


def frameless(*axes):
    for ax in axes:
        ax.spines[:].set_visible(False)


def _pad_linear(a):
    """Pad a 2D coordinate array by one cell on every side, extrapolating linearly."""
    a = np.pad(a, 1, mode="edge").astype(float)
    a[0, :], a[-1, :] = 2 * a[1, :] - a[2, :], 2 * a[-2, :] - a[-3, :]
    a[:, 0], a[:, -1] = 2 * a[:, 1] - a[:, 2], 2 * a[:, -2] - a[:, -3]
    return a


def load_geometry(sample):
    """npz of the main run plus the (1-cell padded) grid -> mask coordinate maps."""
    raw = dict(np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz"))
    shape = (raw["array_row"].max() + 1, raw["array_col"].max() + 1)
    grid_row, grid_col = np.full(shape, np.nan), np.full(shape, np.nan)
    grid_row[raw["array_row"], raw["array_col"]] = raw["mask_row"]
    grid_col[raw["array_row"], raw["array_col"]] = raw["mask_col"]
    raw["shape"], raw["grid_row"], raw["grid_col"] = shape, _pad_linear(grid_row), _pad_linear(grid_col)
    return raw


def _scale_bar(ax, um_per_px, bar_um=1000):
    """Bottom right in display, whatever the axis directions."""
    x0, x1 = ax.get_xlim()
    y1, y0 = ax.get_ylim()
    px = bar_um / um_per_px * np.sign(x1 - x0)
    xe, ye = x1 - 0.05 * (x1 - x0), y1 - 0.05 * (y1 - y0)
    ax.plot([xe - px, xe], [ye, ye], color="black", lw=1.5)
    ax.text(
        xe - px / 2,
        ye - 0.02 * (y1 - y0),
        f"{bar_um / 1000:g} mm",
        ha="center",
        va="bottom",
        bbox=dict(facecolor="white", alpha=0.7, lw=0, pad=1),
    )


def _box_px(raw, box):
    """(x0, x1, y0, y1) in mask px of a crop box (centre row, centre col, half-side um)."""
    r, c, half_um = box
    h = half_um / float(raw["mask_pixel_size_um"])
    return c - h, c + h, r - h, r + h


def _mask_slices(px, pad=2):
    x0, x1, y0, y1 = px
    return slice(max(int(np.floor(y0)) - pad, 0), int(np.ceil(y1)) + pad), slice(
        max(int(np.floor(x0)) - pad, 0), int(np.ceil(x1)) + pad
    )


def _grid_slices(grid_row, grid_col, px, pad=2):
    """Row/column slices of a bin grid whose mask coordinates fall in px, plus pad bins."""
    x0, x1, y0, y1 = px
    inside = (grid_col >= x0) & (grid_col <= x1) & (grid_row >= y0) & (grid_row <= y1)
    rows, cols = np.flatnonzero(inside.any(axis=1)), np.flatnonzero(inside.any(axis=0))
    if len(rows) == 0:
        return slice(0, 0), slice(0, 0)
    return slice(max(rows.min() - pad, 0), rows.max() + pad + 1), slice(max(cols.min() - pad, 0), cols.max() + pad + 1)


def _mask_contour(ax, raw, px, lw):
    """Image-mask outline, cropped to px (nothing outside the panel ends up in the SVG)."""
    mask = raw["mask"]
    rs, cs = _mask_slices(px)
    ax.contour(np.arange(mask.shape[1])[cs], np.arange(mask.shape[0])[rs], mask[rs, cs].astype(float),
               levels=[0.5], colors=[IMAGE_MASK_COLOR], linewidths=lw, rasterized=True)


def _capture_area(ax, raw, px, lw=0.6):
    """Capture-area outline, geometrically clipped to px."""
    from shapely.geometry import LineString, box as shapely_box

    gr, gc = raw["grid_row"][1:-1, 1:-1], raw["grid_col"][1:-1, 1:-1]
    corner = [(0, 0), (0, -1), (-1, -1), (-1, 0), (0, 0)]
    line = LineString([(gc[i, j], gr[i, j]) for i, j in corner]).intersection(shapely_box(px[0], px[2], px[1], px[3]))
    for part in getattr(line, "geoms", [line]):
        if not part.is_empty and part.geom_type == "LineString":
            x, y = part.xy
            ax.plot(x, y, color="black", ls="--", lw=lw)


def analysed_box(sample, raw):
    """Centre (mask px) and half-extent (um) of the analysed bins' bounding box (image mask, qualifying components)."""
    q = pd.read_parquet(
        RESULTS_DIR / f"{sample}_filtered_bins{SUFFIX}.parquet", columns=["components", "mask_row", "mask_col"]
    )
    q = q[q["components"] >= 0]
    (r0, r1), (c0, c1) = q["mask_row"].agg(["min", "max"]), q["mask_col"].agg(["min", "max"])
    half_um = max(r1 - r0, c1 - c0) / 2 * float(raw["mask_pixel_size_um"])
    return (r0 + r1) / 2, (c0 + c1) / 2, half_um


def _crop(ax, raw, box):
    """Square window of half-side box[2] um around (box[0], box[1]) in mask px -- the same physical size in
    every panel of the first row."""
    r, c, half_um = box
    h = half_um / float(raw["mask_pixel_size_um"])
    ax.set_xlim(c - h, c + h)
    ax.set_ylim(r + h, r - h)


def _rotate_180(ax):
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])


def draw_masks(ax, sample, raw, box, rotate=False):
    """Image (H&E / ssDNA) with the image mask (pink), the pipeline's spot-derived footprint (green dotted)
    and the capture area (black dashed), cropped to `box` (see analysed_box)."""
    shape, grid_row, grid_col = raw["shape"], raw["grid_row"], raw["grid_col"]
    spot = pd.read_parquet(
        RESULTS_DIR / f"{sample}_filtered_bins.parquet", columns=["array_row", "array_col", "components"]
    )
    spot = spot[spot["components"] >= 0]
    occupancy = np.zeros((shape[0] + 2, shape[1] + 2))
    occupancy[spot["array_row"] + 1, spot["array_col"] + 1] = 1.0
    px = _box_px(raw, box)
    img, f = raw["image"], int(raw["image_factor"])
    x0, x1, y0, y1 = px
    r0, r1 = max(int(np.floor(y0 / f)) - 1, 0), int(np.ceil(y1 / f)) + 1
    c0, c1 = max(int(np.floor(x0 / f)) - 1, 0), int(np.ceil(x1 / f)) + 1
    crop = img[r0:r1, c0:c1]
    ax.imshow(
        crop,
        cmap="gray" if img.ndim == 2 else None,
        extent=(c0 * f - 0.5, (c0 + crop.shape[1]) * f - 0.5, (r0 + crop.shape[0]) * f - 0.5, r0 * f - 0.5),
        rasterized=True,
    )
    _mask_contour(ax, raw, px, 0.6)
    rs, cs = _grid_slices(grid_row, grid_col, px)
    if occupancy[rs, cs].any():
        ax.contour(
            grid_col[rs, cs],
            grid_row[rs, cs],
            occupancy[rs, cs],
            levels=[0.5],
            colors=[SPOT_MASK_COLOR],
            linewidths=0.8,
            linestyles=":",
            rasterized=True,
        )
    _capture_area(ax, raw, px)
    _crop(ax, raw, box)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if rotate:
        _rotate_180(ax)
    _scale_bar(ax, float(raw["mask_pixel_size_um"]))


def signed_profile(sample, lo_um=-250.0, hi_um=500.0, step_um=8.0, plateau_um=(50.0, 250.0)):
    """Mean raw counts per 8 µm bin (zero-count bins included) vs. signed distance to the image tissue edge
    (negative inside the mask, positive outside), relative to the mean at plateau_um inside."""  # plotted mirrored
    from st_precompute import distance_to_tissue_edge

    raw = dict(np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz"))
    inside = raw["distance_um"] == 0
    signed = raw["distance_um"].astype(float)
    signed[inside] = -distance_to_tissue_edge(raw, inside)
    counts = raw["n_counts"].astype(float)
    plateau = counts[inside & (-signed >= plateau_um[0]) & (-signed <= plateau_um[1])].mean()
    keep = (signed >= lo_um) & (signed < hi_um)
    step = np.floor(signed[keep] / step_um) * step_um + step_um / 2
    prof = pd.Series(counts[keep]).groupby(step).mean() / plateau
    return prof.index.to_numpy(), prof.to_numpy()


def draw_outside(ax, raw, vmax, box, colorbar=True, rotate=False):
    """Raw counts per 8 µm bin outside the image mask over the whole capture area (colour scale 0-vmax),
    image mask pink; cropped to `box` like the image panel."""
    outside = raw["distance_um"] > 0
    counts = np.full(raw["shape"], np.nan)
    counts[raw["array_row"][outside], raw["array_col"][outside]] = raw["n_counts"][outside]
    px = _box_px(raw, box)
    gr, gc = raw["grid_row"][1:-1, 1:-1], raw["grid_col"][1:-1, 1:-1]
    rs, cs = _grid_slices(gr, gc, px)
    mesh = ax.pcolormesh(
        gc[rs, cs],
        gr[rs, cs],
        np.ma.masked_invalid(counts[rs, cs]),
        cmap="YlOrRd",  # as ST_demo
        vmin=0,
        vmax=vmax,
        shading="nearest",
        rasterized=True,
    )
    _mask_contour(ax, raw, px, 0.5)
    _crop(ax, raw, box)
    if rotate:
        _rotate_180(ax)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if colorbar:
        cbar = plt.colorbar(mesh, cax=ax.inset_axes([1.03, 0.1, 0.05, 0.8]), extend="max")
        cbar.ax.tick_params(labelsize=12, length=2, pad=1)
        cbar.set_label("counts", fontsize=12, labelpad=1)


with plt.rc_context({}):
    fig = plt.figure(figsize=(9, 8), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01, wspace=0.02, hspace=0.02)
    top, bottom = fig.subfigures(2, 1, height_ratios=[1, 1.45])
    ax_a, ax_a2, ax_b, ax_b2 = top.subplots(1, 4, width_ratios=[1, 1, 1, 1.12])
    ax_p, ax_c, ax_d, ax_e = bottom.subplots(1, 4, width_ratios=[1.2, 1.0, 0.9, 1.0])

    # a, b: image vs. spot-derived mask, and the raw counts outside the image mask (whole capture area)
    # one colour scale for both count maps: the larger of the two samples' 99th percentiles outside the mask
    geos = [load_geometry(ex[0]) for ex in MASK_EXAMPLES]
    # one window for all four panels: the larger of the two sections' analysed extents, plus 8%
    boxes = [analysed_box(ex[0], g) for ex, g in zip(MASK_EXAMPLES, geos, strict=True)]
    half_um = 1.08 * max(b[2] for b in boxes)
    boxes = [(r, c, half_um) for r, c, _ in boxes]
    vmax = max(np.percentile(g["n_counts"][g["distance_um"] > 0], 99) for g in geos)
    for (ax, ax_out), (sample, tech, stain, rotate), raw, box in zip(
        ((ax_a, ax_a2), (ax_b, ax_b2)), MASK_EXAMPLES, geos, boxes, strict=True
    ):
        draw_masks(ax, sample, raw, box, rotate)
        ax.set_title(f"{sample}\n{stain}")
        draw_outside(ax_out, raw, vmax, box, colorbar=ax_out is ax_b2, rotate=rotate)
        ax_out.set_title("counts\noutside")
    del geos

    # c: count profile across the image tissue edge, all paired samples (Visium HD solid, STOmics dashed)
    for vis, sto in PATIENT_PAIRS:
        for sample, ls in ((vis, "-"), (sto, "--")):
            x, y = signed_profile(sample)
            ax_p.plot(-x, y, color=sample_pal[sample], ls=ls, lw=0.8)  # inside to the right, as ST_demo panel b
    ax_p.axvline(0, color="gray", lw=0.6)
    ax_p.axhline(1, color="gray", lw=0.6, ls=":")
    ax_p.set_yscale("log")
    ax_p.set_xlim(-500, 250)
    ax_p.set_xlabel("distance to edge (µm)\n← outside | inside →")
    ax_p.set_ylabel("relative counts")
    ax_p.set_title("edge profiles")

    # d: tissue-edge elbows per pair, linear axis with "none" below 0 (as ST_demo panel d)
    ZERO_POS, SEP = -30.0, -15.0
    pairs = ["_".join(p) for p in PATIENT_PAIRS]
    d = part2_df.assign(
        x=part2_df["pair"].map({p: i for i, p in enumerate(pairs)})
        + part2_df["technology"].map({"Visium HD": -0.15, "STOmics": 0.15}),
        elbow_plot=part2_df["bsp_thresh_um"].fillna(ZERO_POS),
    )
    kw = dict(
        x="x",
        y="elbow_plot",
        hue="sample",
        style="technology",
        palette=sample_pal,
        size="component_size_um",
        sizes=(15, 220),
        size_norm=(d["component_size_um"].min(), d["component_size_um"].max()),
        markers=DATASET_TYPE_MARKERS,
        hue_order=list(sample_pal.keys()),
    )
    scatter_by_status(ax_c, d, linewidth=0.8, **kw)
    max_val = np.nanmax(d["bsp_thresh_um"])
    ticks = np.arange(0, max_val + 50, 50)
    ax_c.set_yticks([ZERO_POS, *ticks], ["none", *[f"{t:g}" for t in ticks]])
    ax_c.set_ylim(ZERO_POS * 1.4, max_val * 1.05)
    ax_c.axhline(SEP, color="gray", ls=":", lw=0.8)
    ax_c.set_xticks(range(len(pairs)), [f"Pat{i + 1}" for i in range(len(pairs))])
    ax_c.set_xlim(-0.5, len(pairs) - 0.5)
    ax_c.set_xlabel("")
    ax_c.set_ylabel("elbow (µm)")
    ax_c.set_title("elbows")

    # e: % of raw counts outside the image mask, one point per sample
    r = raw_pooled_paired_df.assign(x=raw_pooled_paired_df["pair"].map({p: i for i, p in enumerate(pairs)}))
    sns.scatterplot(
        r,
        x="x",
        y="perc_outside",
        hue="sample",
        style="dataset_type",
        palette=sample_pal,
        s=45,
        ax=ax_d,
        legend=False,
        markers=DATASET_TYPE_MARKERS,
        hue_order=list(sample_pal.keys()),
    )
    ax_d.set_xticks(range(len(pairs)), [f"Pat{i + 1}" for i in range(len(pairs))])
    ax_d.set_xlim(-0.5, len(pairs) - 0.5)
    ax_d.set_ylim(bottom=0)
    ax_d.set_xlabel("")
    ax_d.set_ylabel("% counts outside")
    ax_d.set_title("RNA outside")

    # f: diffusion fit outside the image mask, all samples (demo grey)
    sns.scatterplot(
        raw_pooled_df,
        x="1/beta",
        y="alpha",
        hue="sample",
        palette=sample_pal,
        s=45,
        style="dataset_type",
        ax=ax_e,
        legend=False,
        markers=DATASET_TYPE_MARKERS,
    )
    ax_e.set_xlabel("decay length 1/β (µm)")
    ax_e.set_ylabel("amplitude α (counts/µm²)")
    ax_e.set_title("diffusion fit")

    handles = []
    for vis, sto in PATIENT_PAIRS:
        handles += [
            Line2D([0], [0], marker="s", ls="", color=sample_pal[vis], markersize=6, label=f"{vis} (Visium HD)"),
            Line2D([0], [0], marker="^", ls="", color=sample_pal[sto], markersize=6, label=f"{sto} (STOmics)"),
        ]
    handles += [
        Line2D([0], [0], marker="o", ls="", color="lightgray", markersize=6, label="Visium HD demo"),
        Line2D([0], [0], marker="o", ls="", color="dimgray", markersize=6, label=f"elbow p ≤ {ALPHA}"),
        Line2D([0], [0], marker="o", ls="", markerfacecolor="none", markeredgecolor="dimgray", markersize=6,
               label=f"elbow p > {ALPHA} (dropped)"),
        Line2D([0], [0], marker="X", ls="", color="dimgray", markersize=6, label="elbow untested (< ~0.8 mm)"),
        Line2D([0], [0], color=IMAGE_MASK_COLOR, lw=1, label="image-derived mask"),
        Line2D([0], [0], color=SPOT_MASK_COLOR, lw=1.2, ls=":", label="spot-derived mask"),
        Line2D([0], [0], color="black", lw=0.8, ls="--", label="capture area"),
        Line2D([0], [0], color="gray", lw=0.8, ls="-", label="profile: Visium HD"),
        Line2D([0], [0], color="gray", lw=0.8, ls="--", label="profile: STOmics"),
    ]
    bottom.legend(
        handles=handles, loc="outside lower center", ncol=3, frameon=False, columnspacing=1.0, handletextpad=0.3
    )
    panel_titles(zip([ax_a, ax_b, ax_p, ax_c, ax_d, ax_e], "ABCDEF", strict=True), others=[ax_a2, ax_b2])
    frameless(ax_a, ax_a2, ax_b, ax_b2)
    plt.savefig("ST_figure_paired.svg", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# One bosperrus.plot_fit per sample: the pooled diffusion fit (ConstantFit vs.
# ExponentialDecayFit on every raw 8um bin outside the mask, zero-count bins
# included -- the fit behind panels e/f) over the per-distance-bin quantile bands and mean.
#
# The curve uses the fitted per-bin amplitude a (params["exponential_decay_a"]),
# not alpha: alpha = a / bin area is the cross-resolution-comparable number, but
# the plot shows counts per bin.
#
# Shared distance bin edges (log-spaced) across all subplots. n_counts+1 so zero-count
# bins (most bins far from the tissue) stay visible on a log axis.
all_distance, all_counts = [], []
for sample in raw_pooled_paired_df["sample"]:
    pts = np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz")
    outside = pts["distance_um"] > 0
    all_distance.append(pts["distance_um"][outside])
    all_counts.append(pts["n_counts"][outside])
all_distance, all_counts = np.concatenate(all_distance), np.concatenate(all_counts) + 1

N_BINS = 60
x_edges = np.logspace(np.log10(all_distance.min()), np.log10(all_distance.max()), N_BINS + 1)
del all_distance, all_counts

N_COLS = 4
N_ROWS = int(np.ceil(len(raw_pooled_paired_df) / N_COLS))
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(15, 5.3 * N_ROWS), sharex=True, sharey=True, squeeze=False)
for ax in axes.flat[len(raw_pooled_paired_df) :]:
    ax.set_visible(False)
for ax, (_, row) in zip(axes.flat, raw_pooled_paired_df.iterrows(), strict=False):  # spare axes hidden
    pts = np.load(RESULTS_DIR / f"{row['sample']}_diffusion_8um.npz")
    distance, n_counts = pts["distance_um"], pts["n_counts"]
    outside = distance > 0

    params = row["params"]
    if row["best_fit_type"] == "Exponential Decay Fit":
        a, b = params["exponential_decay_a"], params["exponential_decay_b"]

        def predict_fn(d, a=a, b=b):
            return a * np.exp(-b * d) + 1  # +1 to match the n_counts+1 shift below

    else:  # ConstantFit baseline won instead
        c = params["constant_c"] + 1

        def predict_fn(d, c=c):
            return np.full_like(d, c)

    plot_fit(
        ax,
        distance[outside],
        n_counts[outside] + 1,
        predict_fn,
        bins=x_edges,
        line_kwargs={"color": FIT_PALETTE.get(row["best_fit_type"], "red")},
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{row['sample']} ({row['dataset_type']})", fontsize=12)
    ax.set_xlabel("distance outside mask (um)")
    ax.set_ylabel("n_counts + 1 (raw, per 8um bin)")
    ax.set_ylim(bottom=0.8)  # log axis: n_counts + 1 >= 1
plt.tight_layout()
plt.show()

# Part 4: resolution ablation of the diffusion fit

Same `quantify_diffusion_from_raw` fit repeated at every bin size each technology supports
(`st_precompute.py --ablation`): Visium HD 2/8/16um; STOmics 2/4/8/10/16/25um (bin1 pooled).
If the decay is resolved, `beta` should not depend on bin size, and `alpha` (amplitude per um^2)
only weakly (averaging an exponential over a bin of size s rescales it by ~(beta*s/2)/sinh(beta*s/2)).

In [ ]:
ablation_df = pd.DataFrame([json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob("*_diffusion_*um.json"))])
ablation_df = ablation_df.merge(SAMPLES[["name"]], left_on="sample", right_on="name").sort_values(
    ["sample", "resolution_um"]
)

# stability relative to the 8um value used everywhere above
ref = ablation_df[ablation_df["resolution_um"] == 8.0].set_index("sample")[["alpha", "beta", "perc_counts_outside"]]
for col in ["alpha", "beta", "perc_counts_outside"]:
    ablation_df[f"{col}_rel_8um"] = ablation_df[col] / ablation_df["sample"].map(ref[col])

ablation_df.pivot_table(index=["technology", "sample"], columns="resolution_um", values="beta_rel_8um").round(2)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
for sample, sub in ablation_df.groupby("sample"):
    marker = DATASET_TYPE_MARKERS[sub["dataset_type"].iloc[0]]
    kwargs = dict(color=sample_pal.get(sample, "gray"), marker=marker, label=sample, lw=1)
    axs[0].plot(sub["resolution_um"], sub["beta"], **kwargs)
    axs[1].plot(sub["resolution_um"], sub["alpha"], **kwargs)
    axs[2].plot(sub["resolution_um"], sub["perc_counts_outside"], **kwargs)

for ax, ylabel in zip(
    axs, ["$\\beta$ (1/um)", "$\\alpha$ (counts/um$^2$)", "% counts outside image mask"], strict=True
):
    ax.axvline(8, color="gray", ls="--", lw=1)
    ax.set_xscale("log")
    ax.set_xticks(sorted(ablation_df["resolution_um"].unique()))
    ax.set_xticklabels([f"{r:g}" for r in sorted(ablation_df["resolution_um"].unique())])
    ax.set_xlabel("bin size (um)")
    ax.set_ylabel(ylabel)
axs[0].set_yscale("log")
axs[1].set_yscale("log")
axs[0].set_title("decay rate vs. resolution")
axs[1].set_title("amplitude per area vs. resolution")
axs[2].set_title("counts outside vs. resolution")
axs[2].legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=12)
plt.tight_layout()
plt.show()

# BLADE vs. bosperrus on single cores (image-mask footprint)

Peel-sweep illustration for section 2 (BLADE stops at the first non-significant layer), shown on two
paired-sample cores.

In [ ]:
# BLADE vs. bosperrus on single TMA cores (image-mask footprint): BLADE stops at the
# first non-significant peel layer, even when many significant layers follow;
# bosperrus' elbow comes from one piecewise-linear fit over all bins.
# The elbow is a Euclidean distance (um); peel layers are 8-neighbour steps of 8um,
# so elbow_um / 8 is an approximate position on the layer axis.
PEEL_EXAMPLES = [("Pat2", SUFFIX, "0"), ("Pat4", SUFFIX, "2")]  # tissue-edge run, as panel d
BLADE_COLOR = json.loads(Path("../../fit_palette.json").read_text())["Manual buffer"]  # as in the MICrONS figure
P_FLOOR = 1e-20  # y-axis floor: the outermost layers reach p ~ 1e-283 and would squash the region around 0.05

fig, axs = plt.subplots(1, len(PEEL_EXAMPLES), figsize=(5 * len(PEEL_EXAMPLES), 3.5), sharey=True, squeeze=False)
for ax, (sample, suffix, comp) in zip(axs[0], PEEL_EXAMPLES, strict=True):
    info = json.loads((RESULTS_DIR / f"{sample}_buffer{suffix}.json").read_text())["per_component"][comp]
    sweep = pd.DataFrame(info["blade_sweep"])
    blade_stop = info["blade_thresh"] + 1  # first layer with p >= 0.05 (kept by BLADE)
    elbow_layers = info["elbow_um"] / 8.0 if info["elbow_um"] is not None else np.nan  # None: no rising elbow

    p_shown = sweep["p_value"].clip(lower=P_FLOOR)
    clipped = sweep["p_value"] < P_FLOOR
    sig = sweep["p_value"] < 0.05
    ax.plot(sweep["layer"], p_shown, color="gray", lw=1, zorder=1)
    ax.scatter(
        sweep.loc[sig & ~clipped, "layer"], p_shown[sig & ~clipped], s=12, color="black", zorder=2, label="p < 0.05"
    )
    ax.scatter(
        sweep.loc[clipped, "layer"],
        p_shown[clipped],
        s=22,
        marker="v",
        color="black",
        zorder=2,
        label=f"p < {P_FLOOR:.0e} (clipped)",
    )
    ax.scatter(
        sweep.loc[~sig, "layer"], p_shown[~sig], s=12, facecolor="white", edgecolor="black", zorder=2, label="p >= 0.05"
    )
    ax.axhline(0.05, color="gray", ls="--", lw=1)
    ax.axvspan(0.5, blade_stop - 0.5, color=BLADE_COLOR, alpha=0.15, lw=0, zorder=0)
    ax.axvline(
        blade_stop, color=BLADE_COLOR, lw=1.5, label=f"BLADE stops (buffer {info['blade_thresh_um']:.0f} $\\mu m$)"
    )
    if np.isfinite(elbow_layers):
        ax.axvline(
            elbow_layers,
            color=FIT_PALETTE["Piecewise Linear Fit"],
            lw=1.5,
            label=f"bosperrus elbow ({info['elbow_um']:.0f} $\\mu m$)",
        )
    ax.set_yscale("log")
    ax.set_ylim(P_FLOOR / 3, 1.5)
    ax.set_xlabel("peel iteration (8 $\\mu m$ layers from border)")
    ax.set_title(f"{sample} core {comp} ({info['n_spots']:,} bins)", fontsize=12)
    ax.legend(fontsize=12, loc="lower right")
axs[0, 0].set_ylabel("Welch t-test p (layer vs. interior)")
plt.tight_layout()
plt.savefig("ST_blade_vs_bosperrus_peel.svg", bbox_inches="tight")
plt.show()